<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## 0. What these are for

[Structural](02_Structural_Diagrams.ipynb) and [behavioural](03_Behavioural_Diagrams.ipynb) diagrams
describe a system that exists. The diagrams here describe one that does not exist yet, and their real
job is **agreeing on the boundary before anyone writes code**.

That makes them the most political diagrams in the set. Nobody argues about which port Postgres listens
on. Everybody argues about whether bill disputes are in scope for version one. A scope diagram is
useful precisely to the extent that it makes such an argument happen early and in one room.

| Diagram | Audience | Answers |
|---|---|---|
| **Use case** | Analysts, stakeholders | Who are the actors, and what is inside the boundary? |
| **User flow** | Product, design, engineering | What screens and decisions does a person move through? |
| **Journey map** | Product, leadership | Where in the experience is the pain? |
| **Requirement diagram** | Regulated or safety-critical work | Which requirement does this component satisfy, and how is it verified? |

::: {.callout-warning}
These are the diagram types most often produced as ceremony: drawn once for a document, approved, and
never opened again. Each section below states what the diagram has to contain to avoid that fate. If
yours cannot meet that bar, write a bulleted list of scope and non-scope instead. Nobody will mind.
:::

The running example continues: *MeterHub*, a service that ingests smart-meter readings and bills
customers monthly.

---

## 1. Use case diagram

**What it is:** actors (people or external systems) outside a boundary box, use cases (verb phrases)
inside it, and lines between them. Plus two relationships worth knowing: `<<include>>` for a step that
always happens as part of another, and `<<extend>>` for a conditional variation.

**The honest assessment:** as a description of behaviour it is nearly worthless, because "Pay invoice"
in an oval tells you nothing that the phrase alone did not. It has exactly two genuine uses:

1. **The boundary.** The box is the system. Everything outside it is somebody else's problem, and
   drawing the box forces that conversation.
2. **The actor inventory.** Actors include non-humans, and the external systems are what people forget
   to budget for. The payment gateway, the regulator's reporting portal and the meter firmware are all
   integration work that nobody scoped because they were not in the user stories.

**What makes it non-ceremonial:** an explicit **out of scope** list on the same page. A use case diagram
without one is a wish list.

Mermaid has no use case notation. A flowchart with a subgraph as the boundary carries the same content:

```mermaid
flowchart LR
    CUST(["Customer"])
    OPS(["Billing operator"])
    TECH(["Field technician"])
    GW(["Payment gateway"])
    REG(["Regulator portal"])
    FW(["Meter firmware"])

    subgraph SYS["MeterHub, release 1"]
        UC1(["View usage"])
        UC2(["Download invoice"])
        UC3(["Pay invoice"])
        UC4(["Run billing"])
        UC5(["Correct a reading"])
        UC6(["Commission a meter"])
        UC7(["Authenticate"])
        UC8(["Export regulatory report"])
        UC9(["Ingest reading"])
    end

    CUST --- UC1
    CUST --- UC2
    CUST --- UC3
    OPS --- UC4
    OPS --- UC5
    OPS --- UC8
    TECH --- UC6
    FW --- UC9
    UC3 --- GW
    UC8 --- REG

    UC1 -.->|"include"| UC7
    UC2 -.->|"include"| UC7
    UC3 -.->|"include"| UC7
    UC5 -.->|"extend"| UC4

    classDef actor fill:#2d3748,stroke:#a0aec0,color:#fff
    class CUST,OPS,TECH,GW,REG,FW actor
```

**Explicitly out of scope for release 1:** dispute workflow, tariff self-service switching, prepayment
meters, anything to do with solar export credits.

That last line is the part with value. The diagram gets three actors onto the page that a
requirements document written from customer interviews would have missed entirely, since the payment
gateway, the regulator portal and the meter firmware do not attend interviews.

### PlantUML, if you want the real notation

```plantuml
@startuml
left to right direction
actor Customer
actor "Billing operator" as Ops
rectangle "MeterHub, release 1" {
  usecase UC1 as "View usage"
  usecase UC3 as "Pay invoice"
  usecase UC7 as "Authenticate"
  UC1 ..> UC7 : <<include>>
  UC3 ..> UC7 : <<include>>
}
Customer --> UC1
Customer --> UC3
Ops --> UC1
@enduml
```

---

## 2. User flow

**What it is:** the path a person takes through screens and decisions to finish one task. Not UML,
which is why it survived: product and design actually use these.

**When it earns its keep:** any flow with branching where the branches are *states of the user*, not
states of the code. Signup, checkout, onboarding, password reset, and every flow that can dead-end.

**What separates it from an [activity diagram](03_Behavioural_Diagrams.ipynb):** the nodes are things
the user sees, and the edges are things the user does. If a node is "validate token", it belongs in an
activity diagram instead.

**What makes it non-ceremonial:** every dead end is drawn, and every dead end has an exit. The value of
a user flow is almost entirely in the unhappy branches, because that is where people abandon.

```mermaid
flowchart TD
    START(["Email: your bill is ready"]) --> LAND["Invoice landing page"]
    LAND --> AUTH{"Signed in?"}
    AUTH -->|no| LOGIN["Sign in"]
    LOGIN --> MAGIC{"Password<br/>remembered?"}
    MAGIC -->|no| RESET["Email a magic link"]
    RESET --> INBOX(["User leaves for their inbox"])
    INBOX -.->|"41% never return"| DROP(["Abandoned"])
    MAGIC -->|yes| VIEW
    AUTH -->|yes| VIEW["Invoice detail"]

    VIEW --> ACT{"What now?"}
    ACT -->|"looks wrong"| DISPUTE["No dispute flow yet:<br/>mailto support"]
    DISPUTE --> DROP
    ACT -->|"download"| PDF(["PDF opens"])
    ACT -->|"pay"| METHOD{"Saved card?"}

    METHOD -->|yes| CONFIRM["Confirm amount"]
    METHOD -->|no| CARD["Enter card details"]
    CARD --> CONFIRM
    CONFIRM --> GW["Gateway 3DS challenge"]
    GW -->|"approved"| DONE(["Receipt shown, invoice settled"])
    GW -->|"declined"| FAIL["Decline reason,<br/>retry or change card"]
    FAIL --> METHOD
    GW -->|"timeout"| PENDING["Payment pending:<br/>we will email you"]
    PENDING --> DONE

    classDef bad fill:#742a2a,stroke:#e53e3e,color:#fff
    class DROP,DISPUTE bad
```

Two things fall out of drawing it. The magic-link path hands the user to their email client and 41% of
them never come back, which is a measured number sitting on an edge where it can drive a decision. And
"looks wrong" has no product behind it, so it exits to a mailto link. That was in the out-of-scope list
in section 1, and here is what it costs.

::: {.callout-tip}
Put real numbers on the edges once you have them. A user flow annotated with drop-off rates stops being
a design artifact and becomes an analytics dashboard with a memory.
:::

---

## 3. Journey map

**What it is:** stages of an experience along the horizontal, with a satisfaction score and the actor
at each step. It deliberately says nothing about implementation. Its whole purpose is to make the
emotional low points visible next to each other.

**When it earns its keep:** prioritisation conversations. A journey map is the cheapest way to show a
room that the worst moment in the product is not the feature they were about to fund.

Mermaid has first-class support. Scores run 1 (miserable) to 5 (delighted), and the names after the
score are the actors involved:

```mermaid
journey
    title Customer's first quarter with MeterHub
    section Sign up
      Enter NMI and address: 3: Customer
      Wait for meter commissioning: 2: Customer, Technician
      Confirmation email arrives: 4: Customer
    section First month
      Check live usage daily: 5: Customer
      Notice a gap in the data: 1: Customer
      Email support about the gap: 1: Customer, Support
      Support backfills the reading: 3: Customer, Support
    section First bill
      Bill arrives, no surprises: 4: Customer
      Pay with saved card: 5: Customer
    section Second bill
      Bill is 40 percent higher: 1: Customer
      Look for an explanation in app: 2: Customer
      Give up, phone support: 1: Customer, Support
```

The shape of that map is the argument. Signup is fine, the live usage view is loved, and both troughs
are the same thing wearing different clothes: **when the number is surprising, the product has no
answer**. A data gap and a bill jump both end with the customer phoning a human. That points at one
feature (explain the number in the app) instead of two, and no engineering diagram would have shown it.

::: {.callout-note}
A journey map is only honest if the scores come from users rather than from the team's guesses. Run it
off support tickets, session recordings or five interviews. An invented journey map is a mirror.
:::

---

## 4. Requirement diagram

**What it is:** SysML's traceability notation. Requirements as boxes carrying an id, a risk level and a
**verification method**, linked to each other and to the elements that satisfy or verify them.

**When it earns its keep:** regulated, safety-critical, contractual or certification work, where the
question "prove this requirement is met" gets asked by an auditor. In ordinary product work an issue
tracker does this better.

**What makes it non-ceremonial:** every requirement has a `verifymethod` that names a real artifact.
"Inspection" that means "someone looked at it once" is how traceability matrices become fiction.

```mermaid
requirementDiagram

requirement meter_accuracy {
    id: "REQ-1"
    text: Billing shall use readings accurate to within one percent
    risk: high
    verifymethod: test
}

functionalRequirement gap_handling {
    id: "REQ-1.1"
    text: Periods with more than two percent missing intervals shall not auto bill
    risk: high
    verifymethod: test
}

performanceRequirement ingest_rate {
    id: "REQ-2"
    text: Ingest shall sustain ten thousand readings per second per node
    risk: medium
    verifymethod: demonstration
}

interfaceRequirement mqtt_tls {
    id: "REQ-3"
    text: All meter traffic shall use TLS 1.2 or later with client certificates
    risk: high
    verifymethod: inspection
}

designConstraint retention {
    id: "REQ-4"
    text: Raw interval data shall be retained for seven years
    risk: medium
    verifymethod: analysis
}

element billing_svc {
    type: service
    docref: meterhub_billing
}

element ingest_svc {
    type: service
    docref: meterhub_ingest
}

element gap_test_suite {
    type: test_suite
    docref: tests_test_gap_policy
}

element load_rig {
    type: harness
    docref: bench_mqtt_flood
}

meter_accuracy - contains -> gap_handling
billing_svc - satisfies -> gap_handling
gap_test_suite - verifies -> gap_handling
ingest_svc - satisfies -> ingest_rate
load_rig - verifies -> ingest_rate
ingest_svc - satisfies -> mqtt_tls
ingest_svc - satisfies -> retention
```

Read it as a gap finder rather than a picture. `mqtt_tls` and `retention` are both satisfied by a
service and verified by nothing, and `retention` is a seven-year promise checked by "analysis". Those
two holes are the entire reason to draw the diagram, and they take about four seconds to spot here
against an afternoon in a spreadsheet.

**Relationship vocabulary:** `contains`, `derives`, `satisfies`, `verifies`, `refines`, `traces`,
`copies`. **Requirement types:** `requirement`, `functionalRequirement`, `performanceRequirement`,
`interfaceRequirement`, `physicalRequirement`, `designConstraint`.

::: {.callout-warning}
Quote any `id` that is not a bare word or number. `id: REQ-1` is a parse error, because the hyphen ends
the token and Mermaid then finds text where it wanted a newline. `id: "REQ-1"` renders. The same applies
to `docref`, so keep paths out of it or quote them.
:::

---

## 5. Choosing, and the ceremony test

**Decision shortcut**

| The question in the room | Draw | Skip it if |
|---|---|---|
| "Is that in version one?" | Use case, with an out-of-scope list | You can write the list without the ovals, which is usually |
| "Where do people give up?" | User flow with drop-off numbers | Every branch is happy path |
| "What should we build next?" | Journey map from real research | The scores are guesses |
| "How do we prove we meet this?" | Requirement diagram | No auditor is coming, use the issue tracker |

**The ceremony test.** Before drawing any diagram in this notebook, answer one question: *what decision
changes depending on what this shows?* If the answer is "none, it goes in the document", do not draw
it. Scope diagrams are the ones most likely to be produced because a template has a slot for them, and
a template is not a decision.

**Three things worth more than any of these diagrams**

1. **A written non-scope list.** Two columns, in scope and out of scope, dated and agreed. It takes ten
   minutes and settles more arguments than every use case oval ever drawn.
2. **Acceptance criteria in the tracker.** Given, when, then, attached to the work item, executable
   where possible. Traceability that lives where the work lives.
3. **A prototype.** For anything user-facing, a clickable prototype outruns a user flow diagram, because
   people react to what they can touch and merely nod at what they can read.

Once the boundary is agreed, the next question is how the thing is built. That is
[Architectural Diagrams](05_Architectural_Diagrams.ipynb).

---